# Phase 1 — SBERT Pure Text Representation Experiment (Group K-Fold Cross-Validation)

Eksperimen representasi teks berbasis Sentence-BERT (SBERT `all-MiniLM-L6-v2` 384 dimensi, Murni Teks / PureText tanpa fitur numerik bisnis) untuk prioritasi kebutuhan perangkat lunak (Functional Requirements dan Non-Functional Requirements). Notebook ini mencakup ekstraksi matriks SBERT (384 dimensi embedding), integrasi tipe kebutuhan (FR/NFR), pembentukan query group per proyek, **pembagian data berbasis Group K-Fold Cross-Validation (project_id)**, pelatihan model ranking LightGBM LambdaRank per fold, evaluasi metrik (NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau) dengan agregasi Mean ± Std Dev, serta pencatatan artefak dan metrik ke MLflow server.

In [1]:
%pip install mlflow lightgbm joblib scikit-learn

import logging
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
import json
import joblib
import warnings
import itertools
import requests

try:
    from IPython.display import display
except ImportError:
    display = print

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr, kendalltau
import lightgbm as lgb

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
MLFLOW_URI = "https://mlflow.smbgarasibmw.my.id"

# Koneksi ke tracking server MLflow dengan fallback lokal
server_online = False
try:
    resp = requests.get(MLFLOW_URI, timeout=10)
    if resp.status_code == 200:
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment("Phase_1_SBERT_Text_Representation")
        logger.info(f"Successfully connected to MLflow tracking server: {MLFLOW_URI}")
        server_online = True
    else:
        logger.warning(f"MLflow server at {MLFLOW_URI} returned HTTP {resp.status_code}. Using local MLflow store (./mlruns).")
except Exception as e:
    logger.warning(f"MLflow server unreachable ({e}). Using local MLflow store (./mlruns).")

if not server_online:
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("Phase_1_SBERT_Text_Representation")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: C:\Users\EdselSpth\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


2026-10-28 22:05:02,660 - INFO - Successfully connected to MLflow tracking server: https://mlflow.smbgarasibmw.my.id


## Konfigurasi Parameter Eksperimen

Pengaturan path dataset SBERT PureText, direktori keluaran, seed acak, serta jumlah lipatan Group K-Fold Cross-Validation.

In [2]:
# Path dataset SBERT (otomatis mencari di direktori lokal riset-phase-1/dataset atau output_datasets)
DATASET_PATH = "dataset/Dataset_Training_SBert.csv"
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "riset-phase-1/dataset/Dataset_Training_SBert.csv"
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "../dataset/Dataset_Training_SBert.csv"

OUTPUT_DIR = "outputs/phase1_sbert_gkf"
RANDOM_STATE = 42
N_SPLITS = 5
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Using Dataset: {DATASET_PATH}")
logger.info(f"Output directory initialized at: {OUTPUT_DIR}")

2026-10-28 22:05:02,665 - INFO - Using Dataset: dataset/Dataset_Training_SBert.csv


2026-10-28 22:05:02,669 - INFO - Output directory initialized at: outputs/phase1_sbert_gkf


## STEP 1 — Pemuatan Dataset dan Validasi Skema PureText

Memuat dataset SBERT PureText, memeriksa dimensi matriks, keberadaan kolom wajib untuk ranking (`project_id`, `rank`, `type_encoded` / `type_FR`), serta mengurutkan data berdasarkan `project_id`.

In [3]:
logger.info(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)

logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns count: {len(df.columns)}")

# Validasi dan integrasi kolom tipe kebutuhan untuk ranking PureText
if "type_encoded" not in df.columns and "type_FR" in df.columns:
    df["type_encoded"] = df["type_FR"].astype(int)

required_cols = ["project_id", "rank", "type_encoded"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

logger.info("All required columns present for PureText SBERT ranking pipeline.")

# Urutkan berdasarkan project_id agar data per proyek terkelompok rapi
df = df.sort_values(by="project_id").reset_index(drop=True)
logger.info("Dataset sorted by 'project_id' for ranking context.")
display(df.head(3))

2026-10-28 22:05:02,675 - INFO - Loading dataset from dataset/Dataset_Training_SBert.csv...


2026-10-28 22:05:02,721 - INFO - Dataset shape: (952, 393)


2026-10-28 22:05:02,721 - INFO - Columns count: 393


2026-10-28 22:05:02,722 - INFO - All required columns present for PureText SBERT ranking pipeline.


2026-10-28 22:05:02,723 - INFO - Dataset sorted by 'project_id' for ranking context.


,project_id,value,effort,risk,stakeholder_priority,priority_score,rank,embedding_0,embedding_1,embedding_2,...,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,type_FR,type_NFR,type_encoded
0,P1,3,2,1,3,1.6,19,0.008239,0.000238,-0.056190,...,-0.013309,0.064819,-0.006198,0.079291,0.101408,0.058816,-0.058186,1,0,1
1,P1,3,3,5,3,1.0,36,-0.099394,0.070902,-0.076413,...,0.059586,0.075290,0.012730,0.042156,0.108243,0.034755,-0.047610,0,1,0
2,P1,3,3,2,3,1.3,33,-0.047711,0.081955,-0.059387,...,0.029162,0.075734,-0.005834,0.031445,0.085263,0.042927,-0.053820,0,1,0


## STEP 2 — Validasi Kualitas Data

Pemeriksaan kualitas data yang meliputi identifikasi nilai hilang (missing values), duplikasi baris, keabsahan project_id dan nilai rank, serta pengecekan tipe data seluruh kolom.

In [4]:
validation_report = {}

# Cek missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values: {missing_cols.to_dict()}")
else:
    logger.info("No missing values found.")

# Cek duplikasi data
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = int(dup_rows)
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Validasi project_id dan rank
invalid_pid = df["project_id"].isnull().sum() + (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")
else:
    logger.info("All project IDs are valid.")

invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")
else:
    logger.info("All rank values are valid.")

validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    if k != "dtypes":
        logger.info(f"  {k}: {v}")

2026-10-28 22:05:02,746 - INFO - No missing values found.


2026-10-28 22:05:02,775 - INFO - No duplicate rows found.


2026-10-28 22:05:02,777 - INFO - All project IDs are valid.


2026-10-28 22:05:02,777 - INFO - All rank values are valid.


2026-10-28 22:05:02,781 - INFO - === Validation Report ===


2026-10-28 22:05:02,781 - INFO -   missing_values: 0


2026-10-28 22:05:02,781 - INFO -   duplicate_rows: 0


2026-10-28 22:05:02,781 - INFO -   invalid_project_ids: 0


2026-10-28 22:05:02,781 - INFO -   invalid_rank: 0


## STEP 3 — Ekstraksi Matriks Fitur SBERT

Mengambil seluruh kolom representasi teks SBERT dense embeddings yang diawali dengan `embedding_`.

In [5]:
# Ambil kolom representasi SBERT
embed_cols = sorted([c for c in df.columns if c.startswith("embedding_")], key=lambda x: int(x.split("_")[1]))
X_sbert = df[embed_cols].values

logger.info(f"Extracted pre-computed SBERT matrix with {X_sbert.shape[1]} features.")
logger.info(f"SBERT matrix shape: {X_sbert.shape}")

2026-10-28 22:05:02,789 - INFO - Extracted pre-computed SBERT matrix with 384 features.


2026-10-28 22:05:02,790 - INFO - SBERT matrix shape: (952, 384)


## STEP 4 — Penanganan Dimensi Fitur Teks SBERT

SBERT (Sentence-BERT `all-MiniLM-L6-v2`) merupakan representasi vektor kalimat padat (*dense embeddings*) 384 dimensi yang telah memetakan semantik kebutuhan secara kontekstual. Seluruh 384 dimensi SBERT dipertahankan secara utuh tanpa reduksi fitur.

In [6]:
logger.info(f"Using pre-computed SBERT dense embedding features (Dimension={X_sbert.shape[1]})...")

X_text_selected = X_sbert
selected_sbert_cols = embed_cols

logger.info(f"Text feature matrix shape: {X_text_selected.shape}")
logger.info(f"SBERT features count: {len(selected_sbert_cols)}")

2026-10-28 22:05:02,794 - INFO - Using pre-computed SBERT dense embedding features (Dimension=384)...


2026-10-28 22:05:02,796 - INFO - Text feature matrix shape: (952, 384)


2026-10-28 22:05:02,796 - INFO - SBERT features count: 384


## STEP 5 — Representasi Teks Murni (PureText — Tanpa Fitur Numerik Bisnis)

Sesuai tujuan penelitian Phase 1 (Pure Text Representation), eksperimen ini **tidak menyertakan fitur numerik bisnis** (seperti value, effort, risk, stakeholder_priority) guna menguji kapabilitas murni representasi teks SBERT dalam prioritasi kebutuhan perangkat lunak.

In [7]:
logger.info("Phase 1 Experiment: Pure Text SBERT (No business numeric features).")
X_text_dense = X_text_selected
fused_cols = selected_sbert_cols

logger.info(f"Pure text representation dimension: {X_text_dense.shape[1]}")

2026-10-28 22:05:02,800 - INFO - Phase 1 Experiment: Pure Text SBERT (No business numeric features).


2026-10-28 22:05:02,801 - INFO - Pure text representation dimension: 384


## STEP 6 — Integrasi Fitur Tipe Kebutuhan dan Pembentukan Matriks Fitur Akhir X

Mengintegrasikan fitur kategori `type_encoded` (Functional Requirement = 1, Non-Functional Requirement = 0) ke dalam matriks fitur akhir X.

In [8]:
logger.info("Integrating 'type_encoded' feature into final matrix X...")

if "type_encoded" in df.columns:
    X_type = df[["type_encoded"]].values
    X_matrix = np.hstack((X_text_dense, X_type))
    all_feature_cols = fused_cols + ["type_encoded"]
else:
    X_matrix = X_text_dense
    all_feature_cols = fused_cols

X = pd.DataFrame(X_matrix, columns=all_feature_cols)

logger.info("Feature integration completed.")
logger.info(f"  Final DataFrame X shape: {X.shape}")

2026-10-28 22:05:02,806 - INFO - Integrating 'type_encoded' feature into final matrix X...


2026-10-28 22:05:02,806 - INFO - Feature integration completed.


2026-10-28 22:05:02,806 - INFO -   Final DataFrame X shape: (952, 385)


## STEP 7 — Pembentukan Query Group dan Label Relevansi

Mengonversi kolom rank menjadi label relevansi terurut per proyek (0 = prioritas terendah, N-1 = prioritas tertinggi) untuk objektivitas LightGBM LambdaRank dan mendefinisikan kelompok proyek (groups) berdasarkan `project_id`.

In [9]:
logger.info("Transforming ranks into LambdaRank relevance labels...")

# Transformasi rank ke label relevansi per proyek (0 = terendah, nilai lebih tinggi = lebih relevan)
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

y = df["label"].values
groups = df["project_id"].values

group_sizes = pd.Series(groups).value_counts()
num_groups = len(group_sizes)

logger.info(f"Number of query groups (projects): {num_groups}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

logger.info(f"Label validation -> Range: [{y.min()}, {y.max()}], Unique labels count: {len(np.unique(y))}")

2026-10-28 22:05:02,815 - INFO - Transforming ranks into LambdaRank relevance labels...


2026-10-28 22:05:02,824 - INFO - Number of query groups (projects): 30


2026-10-28 22:05:02,827 - INFO - Group size stats:
count     30.000000
mean      31.733333
std       47.641574
min        3.000000
25%       10.000000
50%       18.000000
75%       23.500000
max      231.000000
Name: count, dtype: float64


2026-10-28 22:05:02,827 - INFO - Label validation -> Range: [0, 19], Unique labels count: 20


## STEP 8 — Inisialisasi Group K-Fold Cross-Validation

Membagi dataset menggunakan GroupKFold(n_splits=N_SPLITS) berdasarkan `project_id`. Pendekatan ini menjamin bahwa setiap lipatan (fold) menguji model pada proyek-proyek yang sama sekali belum pernah dilihat pada data latih (zero project overlap), sehingga mengeliminasi data leakage.

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS)
_
logger.info(f"Initialized GroupKFold with {N_SPLITS} splits on project_id.")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    train_projs = set(groups[train_idx])
    val_projs = set(groups[val_idx])
    overlap = train_projs & val_projs
    logger.info(f"Fold {fold}: Train projects={len(train_projs)}, Val projects={len(val_projs)}, Overlap={len(overlap)}")
    if len(overlap) > 0:
        raise ValueError(f"Data leakage detected in Fold {fold}: overlap={overlap}")

logger.info("Zero project overlap verified across all folds!")

2026-10-28 22:05:02,835 - INFO - Initialized GroupKFold with 5 splits on project_id.


2026-10-28 22:05:02,857 - INFO - Fold 1: Train projects=29, Val projects=1, Overlap=0


2026-10-28 22:05:02,857 - INFO - Fold 2: Train projects=26, Val projects=4, Overlap=0


2026-10-28 22:05:02,857 - INFO - Fold 3: Train projects=23, Val projects=7, Overlap=0


2026-10-28 22:05:02,858 - INFO - Fold 4: Train projects=21, Val projects=9, Overlap=0


2026-10-28 22:05:02,858 - INFO - Fold 5: Train projects=21, Val projects=9, Overlap=0


2026-10-28 22:05:02,858 - INFO - Zero project overlap verified across all folds!


## STEP 9 — Pelatihan dan Evaluasi LightGBM LambdaRank per Fold (Cross-Validation)

Melatih model LightGBM Ranker dengan objektivitas lambdarank pada data latih setiap fold dan mengevaluasi metrik ranking (NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau) pada proyek uji yang belum pernah dilihat model.

In [11]:
fold_metrics_list = []
oof_predictions = np.zeros(len(df))
models = []

logger.info(f"Starting {N_SPLITS}-Fold Group Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    logger.info(f"\n{'='*20} Fold {fold}/{N_SPLITS} {'='*20}")
    
    # Wajib sort index berdasarkan project_id agar query group LightGBM terkelompok rapi
    train_idx = sorted(train_idx, key=lambda i: groups[i])
    val_idx = sorted(val_idx, key=lambda i: groups[i])
    
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    train_groups = groups[train_idx]
    val_groups = groups[val_idx]
    
    # Hitung ukuran kelompok (group query counts) per proyek
    train_group_counts = [len(list(g)) for _, g in itertools.groupby(train_groups)]
    val_group_counts = [len(list(g)) for _, g in itertools.groupby(val_groups)]
    
    max_train_group = max(train_group_counts)
    num_leaves = min(max_train_group, 255)
    
    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=num_leaves,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    ranker.fit(
        X_train, y_train,
        group=train_group_counts,
        eval_set=[(X_val, y_val)],
        eval_group=[val_group_counts],
        eval_at=[5, 10],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    models.append(ranker)
    
    # Prediksi skor ranking untuk validasi
    val_pred = ranker.predict(X_val)
    oof_predictions[val_idx] = val_pred
    
    # Hitung metrik per proyek lalu rata-ratakan untuk fold ini
    ndcg5_list, ndcg10_list, map_list = [], [], []
    spearman_list, kendall_list = [], []
    
    idx_start = 0
    for g_len in val_group_counts:
        p_slice = slice(idx_start, idx_start + g_len)
        p_y = y_val[p_slice]
        p_pred = val_pred[p_slice]
        idx_start += g_len
        
        # NDCG (memerlukan 2D array jika menggunakan ndcg_score)
        if len(np.unique(p_y)) > 1:
            try:
                n5 = ndcg_score([p_y], [p_pred], k=5)
                n10 = ndcg_score([p_y], [p_pred], k=10)
                ndcg5_list.append(n5)
                ndcg10_list.append(n10)
            except Exception:
                pass
            
            # Spearman & Kendall Tau
            sp_corr, _ = spearmanr(p_y, p_pred)
            kt_corr, _ = kendalltau(p_y, p_pred)
            if not np.isnan(sp_corr):
                spearman_list.append(sp_corr)
            if not np.isnan(kt_corr):
                kendall_list.append(kt_corr)
        
        # Binarize label untuk MAP
        binary_y = (p_y >= np.median(p_y)).astype(int)
        if len(np.unique(binary_y)) > 1:
            map_score = average_precision_score(binary_y, p_pred)
            map_list.append(map_score)
    
    fold_ndcg5 = np.mean(ndcg5_list) if ndcg5_list else ndcg_score([y_val], [val_pred], k=5)
    fold_ndcg10 = np.mean(ndcg10_list) if ndcg10_list else ndcg_score([y_val], [val_pred], k=10)
    fold_map = np.mean(map_list) if map_list else 0.0
    fold_spearman = np.mean(spearman_list) if spearman_list else spearmanr(y_val, val_pred)[0]
    fold_kendall = np.mean(kendall_list) if kendall_list else kendalltau(y_val, val_pred)[0]
    
    # Hitung p-value keseluruhan fold
    sp_all, sp_pval = spearmanr(y_val, val_pred)
    kt_all, kt_pval = kendalltau(y_val, val_pred)
    
    fold_result = {
        "fold": fold,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_projects": len(train_group_counts),
        "val_projects": len(val_group_counts),
        "NDCG@5": fold_ndcg5,
        "NDCG@10": fold_ndcg10,
        "MAP": fold_map,
        "Spearman": fold_spearman,
        "Spearman_pvalue": sp_pval,
        "KendallTau": fold_kendall,
        "KendallTau_pvalue": kt_pval
    }
    fold_metrics_list.append(fold_result)
    logger.info(f"Fold {fold} Results -> NDCG@5: {fold_ndcg5:.4f}, NDCG@10: {fold_ndcg10:.4f}, MAP: {fold_map:.4f}, Spearman: {fold_spearman:.4f}, KendallTau: {fold_kendall:.4f}")

cv_details_df = pd.DataFrame(fold_metrics_list)
display(cv_details_df)

2026-10-28 22:05:02,871 - INFO - Starting 5-Fold Group Cross-Validation...


2026-10-28 22:05:02,873 - INFO - 
==================== Fold 1/5 ====================


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006154 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 92414
[LightGBM] [Info] Number of data points in the train set: 721, number of used features: 385
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

2026-10-28 22:05:03,145 - INFO - Fold 1 Results -> NDCG@5: 0.6714, NDCG@10: 0.6133, MAP: 0.6479, Spearman: 0.0087, KendallTau: 0.0076


2026-10-28 22:05:03,145 - INFO - 
==================== Fold 2/5 ====================


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

2026-10-28 22:05:05,011 - INFO - Fold 5 Results -> NDCG@5: 0.7147, NDCG@10: 0.7897, MAP: 0.7368, Spearman: 0.2669, KendallTau: 0.2036


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,fold,train_rows,val_rows,train_projects,val_projects,NDCG@5,NDCG@10,MAP,Spearman,Spearman_pvalue,KendallTau,KendallTau_pvalue
0,1,721,231,29,1,0.671400,0.613330,0.647863,0.008741,0.894884,0.007635,0.867508
1,2,775,177,26,4,0.760541,0.840865,0.796925,0.273261,0.058441,0.234206,0.041312
2,3,769,183,23,7,0.782543,0.820696,0.823732,0.410314,0.094817,0.297613,0.097309
3,4,774,178,21,9,0.586316,0.697891,0.670171,0.053293,0.200014,0.033942,0.195285
4,5,769,183,21,9,0.714682,0.789696,0.736824,0.266943,0.000004,0.203596,0.000010


## STEP 10 — Agregasi Metrik Cross-Validation (Mean ± Std Dev)

Menghitung performa agregat di seluruh lipatan pengujian mencakup rata-rata dan standar deviasi dari NDCG@5, NDCG@10, MAP, Spearman, dan Kendall Tau.

In [12]:
metric_cols = ["NDCG@5", "NDCG@10", "MAP", "Spearman", "Spearman_pvalue", "KendallTau", "KendallTau_pvalue"]
cv_summary = {}

for col in metric_cols:
    cv_summary[f"{col}_mean"] = float(cv_details_df[col].mean())
    cv_summary[f"{col}_std"] = float(cv_details_df[col].std())

summary_rows = []
for col in ["NDCG@5", "NDCG@10", "MAP", "Spearman", "KendallTau"]:
    m = cv_summary[f"{col}_mean"]
    s = cv_summary[f"{col}_std"]
    summary_rows.append({
        "Metric": col,
        "Mean ± Std": f"{m:.6f} ± {s:.6f}",
        "Mean": m,
        "Std": s
    })

cv_summary_df = pd.DataFrame(summary_rows)
logger.info("=== Cross-Validation Aggregated Summary ===")
logger.info(f"\n{cv_summary_df[['Metric', 'Mean ± Std']].to_string(index=False)}")
display(cv_summary_df)

2026-10-28 22:05:05,032 - INFO - === Cross-Validation Aggregated Summary ===


2026-10-28 22:05:05,034 - INFO - 
    Metric          Mean ± Std
    NDCG@5 0.703097 ± 0.078084
   NDCG@10 0.752496 ± 0.095123
       MAP 0.735103 ± 0.076660
  Spearman 0.202510 ± 0.167446
KendallTau 0.155398 ± 0.127812


,Metric,Mean ± Std,Mean,Std
0,NDCG@5,0.703097 ± 0.078084,0.703097,0.078084
1,NDCG@10,0.752496 ± 0.095123,0.752496,0.095123
2,MAP,0.735103 ± 0.076660,0.735103,0.076660
3,Spearman,0.202510 ± 0.167446,0.202510,0.167446
4,KendallTau,0.155398 ± 0.127812,0.155398,0.127812


## STEP 11 — Penyimpanan Artefak dan Output Eksperimen

Menyimpan berkas ringkasan metrik CV JSON, tabel detail fold CSV, out-of-fold predictions CSV, daftar fitur, dan model terbaik ke direktori keluaran.

In [13]:
# Simpan model terbaik (berdasarkan NDCG@5)
best_fold_idx = int(cv_details_df["NDCG@5"].argmax())
best_model = models[best_fold_idx]
best_model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker_best_fold.pkl")
joblib.dump(best_model, best_model_path)
logger.info(f"Best model (Fold {best_fold_idx + 1}) saved to: {best_model_path}")

# Simpan CV fold details
cv_details_path = os.path.join(OUTPUT_DIR, "cv_fold_details.csv")
cv_details_df.to_csv(cv_details_path, index=False)
logger.info(f"CV fold details saved to: {cv_details_path}")

# Simpan CV metrics summary JSON
cv_summary_path = os.path.join(OUTPUT_DIR, "cv_metrics_summary.json")
with open(cv_summary_path, "w") as f:
    json.dump(cv_summary, f, indent=2)
logger.info(f"CV metrics summary saved to: {cv_summary_path}")

# Simpan Out-of-Fold (OOF) Predictions
oof_df = pd.DataFrame({
    "project_id": groups,
    "true_label": y,
    "oof_predicted_score": oof_predictions
})
oof_path = os.path.join(OUTPUT_DIR, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)
logger.info(f"OOF predictions saved to: {oof_path}")

# Simpan daftar fitur
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved to: {features_path}")

# Simpan dataset hasil pemrosesan lengkap
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "sbert_puretext_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Processed dataset saved to: {dataset_path} (shape: {dataset_out.shape})")

2026-10-28 22:05:05,056 - INFO - Best model (Fold 3) saved to: outputs/phase1_sbert_gkf\lgbm_ranker_best_fold.pkl


2026-10-28 22:05:05,058 - INFO - CV fold details saved to: outputs/phase1_sbert_gkf\cv_fold_details.csv


2026-10-28 22:05:05,060 - INFO - CV metrics summary saved to: outputs/phase1_sbert_gkf\cv_metrics_summary.json


2026-10-28 22:05:05,064 - INFO - OOF predictions saved to: outputs/phase1_sbert_gkf\oof_predictions.csv


2026-10-28 22:05:05,065 - INFO - Feature list saved to: outputs/phase1_sbert_gkf\feature_list.txt


2026-10-28 22:05:05,363 - INFO - Processed dataset saved to: outputs/phase1_sbert_gkf\sbert_puretext_dataset.csv (shape: (952, 388))


## STEP 12 — Pencatatan MLflow dan Ringkasan Akhir Eksperimen

Mencatat seluruh parameter Group K-Fold, metrik agregasi mean ± std (dengan sanitasi nama metrik agar kompatibel dengan server MLflow), dan artefak ke server MLflow tracking https://mlflow.smbgarasibmw.my.id/ serta menampilkan tabel ringkasan eksperimen.

In [14]:
if mlflow.active_run():
    mlflow.end_run()

try:
    with mlflow.start_run(run_name="SBERT PureText GKF Experiment Run") as run:
        # Log parameter eksperimen
        mlflow.log_param("text_representation", "PureText SBERT")
        mlflow.log_param("embedding_model", EMBEDDING_MODEL_NAME)
        mlflow.log_param("embedding_dimension", X_text_selected.shape[1])
        mlflow.log_param("total_fused_features", X.shape[1])
        mlflow.log_param("validation_strategy", "GroupKFold")
        mlflow.log_param("n_splits", N_SPLITS)
        mlflow.log_param("group_column", "project_id")
        mlflow.log_param("total_projects", len(np.unique(groups)))
        mlflow.log_param("ranker_objective", "lambdarank")
        mlflow.log_param("ranker_n_estimators", 100)
        mlflow.log_param("random_state", RANDOM_STATE)

        # Log metrik agregasi CV (mean dan std) — sanitasi '@' menjadi '_at_' agar valid di MLflow
        mlflow_metrics = {k.replace("@", "_at_"): float(v) for k, v in cv_summary.items()}
        mlflow.log_metrics(mlflow_metrics)

        # Log artefak
        mlflow.log_artifact(best_model_path, artifact_path="model")
        mlflow.log_artifact(cv_summary_path, artifact_path="metrics")
        mlflow.log_artifact(cv_details_path, artifact_path="metrics")
        mlflow.log_artifact(features_path, artifact_path="features")
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        mlflow.log_artifact(oof_path, artifact_path="predictions")

        logger.info(f"MLflow run ID: {run.info.run_id}")
        logger.info("MLflow experiment logged successfully with Group K-Fold metrics.")
except Exception as e:
    logger.warning(f"MLflow logging error: {e}. Experiment execution & local artifacts completed successfully.")

logger.info("=== Experiment Group K-Fold Completed Successfully ===")
display(cv_summary_df)

2026-10-28 22:05:09,094 - INFO - MLflow run ID: 960356531a134de3aa26049b27067af5


2026-10-28 22:05:09,096 - INFO - MLflow experiment logged successfully with Group K-Fold metrics.


🏃 View run SBERT PureText GKF Experiment Run at: https://mlflow.smbgarasibmw.my.id/#/experiments/8/runs/960356531a134de3aa26049b27067af5
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/8


2026-10-28 22:05:09,442 - INFO - === Experiment Group K-Fold Completed Successfully ===


,Metric,Mean ± Std,Mean,Std
0,NDCG@5,0.703097 ± 0.078084,0.703097,0.078084
1,NDCG@10,0.752496 ± 0.095123,0.752496,0.095123
2,MAP,0.735103 ± 0.076660,0.735103,0.076660
3,Spearman,0.202510 ± 0.167446,0.202510,0.167446
4,KendallTau,0.155398 ± 0.127812,0.155398,0.127812
